# Brazil Criminal Events — Bronze Raw Loader

Scans the **unprocessed** volume folder for any Excel file (`.xlsx` / `.xls`), loads each into a raw Delta table in `brazil_criminal_events.bronze` using **overwrite** mode with an `insert_ts` audit column, then moves the file to the **processed** folder.

In [0]:
%pip install openpyxl

In [0]:
import os
import re
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG          = "brazil_criminal_events"
SCHEMA           = "bronze"
UNPROCESSED_PATH = "/Volumes/brazil_criminal_events/landing/unprocessed/"
PROCESSED_PATH   = "/Volumes/brazil_criminal_events/landing/processed/"
SUPPORTED_EXTS   = {".xlsx", ".xls"}

# ---------------------------------------------------------------------------
# Ensure catalog and schema exist
# ---------------------------------------------------------------------------
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# ---------------------------------------------------------------------------
# Helper: derive a valid Delta table name from a file name
#   "BancoVDE_2025.xlsx" -> "banco_vde_2025"
# ---------------------------------------------------------------------------
def file_to_table_name(file_name: str) -> str:
    stem = os.path.splitext(file_name)[0]          # remove extension
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", stem)  # non-alnum -> underscore
    cleaned = re.sub(r"^_+|_+$", "", cleaned)       # strip leading/trailing _
    return cleaned.lower()

# ---------------------------------------------------------------------------
# List files in the unprocessed folder
# ---------------------------------------------------------------------------
files = dbutils.fs.ls(UNPROCESSED_PATH)
excel_files = [
    f for f in files
    if not f.isDir() and os.path.splitext(f.name)[1].lower() in SUPPORTED_EXTS
]

if not excel_files:
    print(f"No Excel files found in {UNPROCESSED_PATH}. Nothing to process.")
else:
    print(f"Found {len(excel_files)} file(s) to process:\n")

    for f in excel_files:
        file_name    = f.name
        src_path     = f.path                             # dbfs:/Volumes/.../file.xlsx
        file_size_mb = f.size / (1024 * 1024)
        table_name   = file_to_table_name(file_name)
        full_table   = f"{CATALOG}.{SCHEMA}.{table_name}"

        print(f"{'=' * 70}")
        print(f"  File : {file_name}  ({file_size_mb:.1f} MB)")
        print(f"  Table: {full_table}")

        # --- 1. Read the Excel file with pandas ------------------------------
        # dbutils.fs paths use the dbfs:/ scheme; pandas needs /Volumes/...
        vol_path = src_path.replace("dbfs:", "")
        pdf = pd.read_excel(vol_path, engine="openpyxl")
        print(f"  Rows : {len(pdf):,}  |  Columns: {len(pdf.columns)}")

        # --- 2. Convert to Spark DataFrame & add audit columns ---------------
        df = spark.createDataFrame(pdf)
        df = df.withColumn("source_file", lit(file_name))
        df = df.withColumn("insert_ts",   current_timestamp())

        # --- 3. Write as a managed Delta table (overwrite) -------------------
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(full_table)
        )
        print(f"  \u2713 Table {full_table} written successfully.")

        # --- 4. Move the file to the processed folder -----------------------
        dest_path = f"{PROCESSED_PATH}{file_name}"
        # Remove destination if it already exists (dbutils.fs.mv has no overwrite on serverless)
        try:
            dbutils.fs.rm(dest_path)
        except Exception:
            pass
        dbutils.fs.mv(src_path, dest_path)
        print(f"  \u2713 Moved to {dest_path}")

    print(f"\n{'=' * 70}")
    print("All files processed.")